# FastF1 | Preprocess the 2019 Silverstone race

Start with three synthetic examples to check feature timing. Then use FastF1's public Jolpica client to build the team dataset for all 20 drivers and inspect Tony's Verstappen subset. The notebook saves the full feature table to `local_results/silverstone_2019_jolpica_features.csv`.

Run from the repository root with the project environment as your notebook kernel. Public race records require network access or a populated FastF1 cache. The optional authenticated session inspection below is controlled by `RUN_LIVE`.

## What to record

Record the race, command or cell, Python/package versions, actual output or exact error, and your interpretation. Link the evidence in the team [progress page](https://app.notion.com/p/41bc0f7b1a7a811ea92ce25a482a8498). Do not treat a stored-data check as a successful fresh run.

## 1. Existing public API calls and practical functions

| Official function / property | What it does | Inputs | Output / caveat |
|---|---|---|---|
| `fastf1.get_event_schedule(year)` | Discover events in a season | Year | EventSchedule table; not lap data |
| `fastf1.get_session(year, gp, identifier)` | Identify one session | Season, event number/name, session code (`R` race, `Q` qualifying) | Session object, not yet loaded |
| `session.load()` | Obtain supported session sources | Session and optional load flags | Populates data properties; download or cache may fail |
| `session.laps` | Driver-lap records | Loaded Session | Laps, DataFrame subclass |
| `session.results` | Session classification/grid metadata | Loaded Session | SessionResults table; most race-result fields are outcomes |
| `session.track_status` | Track status records | Loaded Session | Timestamped flag/status changes |
| `session.weather_data` | Weather observations | Loaded Session | Timestamped weather; not necessarily aligned per lap |
| `Laps.pick_fastest()` | Select fastest lap | Laps | Lap, Series subclass; not an advance feature if computed using future laps |
| `fastf1.Cache.enable_cache(cache_dir)` | Configure cache | Existing writable directory | Caching side effect |
| `Ergast.get_race_results(...)` | Classification API used by team sanity script | Season and round | Response containing classification DataFrame |

Sources: [Getting Started](https://github.com/theOehrly/Fast-F1/blob/main/docs/getting_started/basics.rst); [Timing Data](https://github.com/theOehrly/Fast-F1/blob/main/docs/api_reference/timing_data.rst); [team sanity script](https://github.com/hitotoro24/cs682-f1-meets-ai/blob/d86ff05c56b456fbded2c3581fdc2ae30a219247/race-simulation/experiments/sanity_check.py). The team should independently verify APIs against its pinned version and author its own functions as needed.

## 2. Environment and data access

Install FastF1 in the notebook kernel before running the live cells. Use the team-approved version; the simulator port records `fastf1==3.9.2`, whose availability must be checked in your environment. FastF1 uses different sources for timing and classification; access to results does not imply access to lap telemetry. Do not put credentials in the notebook. If the API fails, keep the exact error and date rather than silently switching sessions.

The next two cells define small inspection functions and call the official API directly. Inputs, return values and limitations are documented at each function. Source: [FastF1 Getting Started](https://docs.fastf1.dev/getting_started/basics.html).

In [1]:
from dataclasses import dataclass


@dataclass(frozen=True)
class CheckResult:
    """Result of one check."""

    name: str
    status: str
    detail: str


def show_results(results: list[CheckResult]) -> None:
    """Print the status and details of each check."""
    for result in results:
        print(f"[{result.status}] {result.name}: {result.detail}")

In [2]:
import importlib.util
import platform
from importlib.metadata import version
from pathlib import Path

# Change only these settings to test another event.
YEAR = 2019
EVENT = "British Grand Prix"
SESSION_TYPE = "R"
DRIVER = "VER"
CACHE_DIR = Path.cwd() / "fastf1_cache"
RUN_LIVE = False  # Enable for the optional authenticated session inspection.


def load_race(year: int, event: str, session_type: str, cache_dir: Path):
    """Load a FastF1 Session via the public API; return the loaded Session.

    Requires installed FastF1 and data/cache access. No feature engineering,
    target generation or model fitting occurs here.
    """
    import fastf1

    cache_dir.mkdir(parents=True, exist_ok=True)
    fastf1.Cache.enable_cache(str(cache_dir))
    session = fastf1.get_session(year, event, session_type)
    session.load(laps=True, telemetry=False, weather=False, messages=True)
    return session


def inspect_driver_laps(laps, driver: str, first_lap: int, last_lap: int):
    """Display raw records for a named driver and inclusive lap interval.

    Inputs: FastF1 Laps table, driver code, first/last lap integers.
    Output: a copy of selected source columns; no new labels are created.
    """
    columns = ["Driver", "LapNumber", "LapStartTime", "Time", "LapTime",
               "PitInTime", "PitOutTime", "Compound", "TyreLife", "Stint",
               "Position", "TrackStatus"]
    missing = [name for name in columns if name not in laps.columns]
    if missing:
        raise ValueError(f"Required source columns missing: {missing}")
    is_driver = laps["Driver"] == driver
    is_selected_lap = laps["LapNumber"].between(first_lap, last_lap)
    selected = laps.loc[is_driver & is_selected_lap, columns]
    return selected.sort_values("LapNumber").copy()

### Small preprocessing example

Check feature timing on three synthetic examples before requesting live race data.

In [3]:
from f1_ai.pit_pipeline import FEATURES, historical_examples, smoke_test

smoke_test()

SYNTHETIC ONLY: feature and no-current-lap-leakage checks passed
 lap  race_progress  position_fraction  previous_pits  pit
   2       0.019231               0.20            0.0    1
   3       0.038462               0.40            1.0    0
   4       0.057692               0.25            1.0    0
PyTorch toy overfit: BCE 0.714567 -> 0.000583; shape (3,)


In [4]:
session = None
source_rows = None
if not RUN_LIVE:
    live_result = CheckResult("FastF1 live load", "SKIPPED", "RUN_LIVE=False")
elif importlib.util.find_spec("fastf1") is None:
    live_result = CheckResult("FastF1 live load", "BLOCKED",
                              "FastF1 is not installed in this kernel.")
else:
    try:
        session = load_race(YEAR, EVENT, SESSION_TYPE, CACHE_DIR)
        laps = session.laps
    except Exception as error:
        live_result = CheckResult("FastF1 live load", "BLOCKED",
                                  f"{type(error).__name__}: {error}")
    else:
        try:
            if laps.empty:
                raise ValueError("FastF1 returned no laps.")
            preview_rows = inspect_driver_laps(laps, DRIVER, 12, 14)
            if preview_rows["LapNumber"].tolist() != [12, 13, 14]:
                raise ValueError("The expected driver-lap records were not returned.")
            source_rows = preview_rows
            live_result = CheckResult("FastF1 live load", "PASS",
                                      f"FastF1 {version('fastf1')}; {len(laps)} raw lap rows;"
                                      f" {session.event['EventName']} {session.name}; {DRIVER} laps 12–14 present.")
        except (AttributeError, KeyError, ValueError, TypeError) as error:
            live_result = CheckResult("FastF1 live load", "FAIL",
                                      f"Source returned but expected checks failed: {error}")

print(f"Kernel: Python {platform.python_version()}")
show_results([live_result])
if source_rows is not None:
    display(source_rows)

Kernel: Python 3.11.9
[SKIPPED] FastF1 live load: RUN_LIVE=False


## 3. Lap-table dictionary (31 documented columns)

| Fields | Type / semantics | Role to discuss |
|---|---|---|
| Driver, DriverNumber, Team | Identity strings | Keys or approved historical identity features |
| LapNumber, Stint | Numeric counters | Progress, stint history |
| Time, LapStartTime | Session-relative timedelta | Lap endpoint/start timestamps; verify chronological boundary |
| LapStartDate | Absolute datetime | Time alignment only |
| LapTime | Timedelta duration | Prior completed performance, not target-lap final duration |
| PitInTime, PitOutTime | Session-relative entry/exit timestamps | Event audit / target construction, not future predictors |
| Sector1Time, Sector2Time, Sector3Time | Sector durations | Optional prior performance |
| Sector1SessionTime, Sector2SessionTime, Sector3SessionTime | Sector timestamps | Within-lap cutoff audit |
| SpeedI1, SpeedI2, SpeedFL, SpeedST | Speed measurements | Optional, not all map into TUM |
| IsPersonalBest | Boolean | Check timing; avoid full-race derived features |
| Compound | Relative dry/wet tyre label | Categorical, race-specific mapping |
| TyreLife | Laps on set including prior-session use | Not necessarily TUM effective degradation age |
| FreshTyre | Boolean set freshness | Stint audit |
| TrackStatus | Status occurring on lap | Can aggregate multiple states; use timestamped source for causal flag state |
| Position | Running position at lap completion | Align cutoff; not final classification |
| Deleted, DeletedReason | Quality/deletion flags | Quality audit |
| FastF1Generated, IsAccurate | Quality markers | Audit missing/inaccurate rows; do not blindly filter pit laps |

`Time` and `LapTime` are different. `session.results.Position` is a different quantity from `session.laps.Position`. Inter-driver gaps require careful common-time alignment; the Laps table does not have gap-ahead/behind columns. Timing streams or telemetry can differ in meaning from a time gap at a lap crossing.

## 4. Inspect the data and schema

These read-only checks describe what the source actually returned. They do not filter the research cohort or construct features and labels. The expected columns are the documented standard lap-table fields; version-specific differences are reported rather than concealed.

In [5]:
EXPECTED_COLUMNS = (
    "Driver", "DriverNumber", "Team", "LapNumber", "Stint", "Time", "LapTime",
    "LapStartTime", "LapStartDate", "PitInTime", "PitOutTime", "Compound", "TyreLife",
    "FreshTyre", "Sector1Time", "Sector2Time", "Sector3Time", "SpeedI1", "SpeedI2",
    "SpeedFL", "SpeedST", "IsPersonalBest", "Sector1SessionTime", "Sector2SessionTime",
    "Sector3SessionTime", "Position", "TrackStatus", "Deleted", "DeletedReason",
    "FastF1Generated", "IsAccurate",
)


def report_schema(actual_columns) -> CheckResult:
    """Compare actual column names against the documented source schema."""
    actual = set(actual_columns)
    missing = sorted(set(EXPECTED_COLUMNS) - actual)
    extra = sorted(actual - set(EXPECTED_COLUMNS))
    status = "REVIEW" if missing else "PASS"
    return CheckResult("Source column coverage", status,
                       f"Missing: {missing or 'none'}; additional: {extra or 'none'}.")


schema_reference = CheckResult("Documented schema", "PASS" if len(EXPECTED_COLUMNS) == 31
                               and len(set(EXPECTED_COLUMNS)) == 31 else "FAIL",
                               f"{len(EXPECTED_COLUMNS)} listed column names.")
show_results([schema_reference])
# Check that a missing-column report is detectable without using fictional race rows.
control = report_schema(EXPECTED_COLUMNS[:-1])
show_results([CheckResult("Schema negative control", "PASS" if control.status == "REVIEW" else "FAIL",
                          "Removed one documented field; missing-column check must flag it.")])

if live_result.status == "PASS":
    show_results([report_schema(laps.columns)])
    print(f"Lap rows: {len(laps):,}; drivers: {laps['Driver'].nunique()}")
    print("Duplicate driver/lap keys:", laps.duplicated(["Driver", "LapNumber"]).sum())
    print("Raw missing counts (selected fields):")
    display(laps[["LapTime", "Compound", "TyreLife", "Position", "PitInTime"]].isna().sum())
    print("Note: missing PitInTime is expected for non-pit laps; it is not ordinary missingness.")
    print("Session result table:", session.results.shape)
    display(session.results.head())
else:
    show_results([CheckResult("Live source schema and row checks",
                              "SKIPPED" if live_result.status == "SKIPPED" else "BLOCKED",
                              "No successful live session load; no historical rows asserted.")])

[PASS] Documented schema: 31 listed column names.
[PASS] Schema negative control: Removed one documented field; missing-column check must flag it.
[SKIPPED] Live source schema and row checks: No successful live session load; no historical rows asserted.


## 5. Check one historical pit event

Use the displayed VER laps 12–14 from the live check above. The TUM configuration records an inlap on 13, with compound `A3` fitted. Compare the `PitInTime`, `PitOutTime`, `Compound` and `Stint` fields. If timing is unavailable, report that blocker. The notebook does not use a synthetic substitute or infer successful acquisition from TUM's recorded strategy.

The raw observation is not a feature vector. The team must agree on which event time defines the target, which predictors are available before that target, and whether the corresponding variable exists in the TUM simulated state.

## 6. Proposed function boundaries

The following are work packages to claim, not existing functions or assignment solutions. Agree on names and signatures, then write independently under course policy. Each function should have a documented purpose, inputs, return schema, units, assumptions, exceptions and smallest test.

| Responsibility | Input to establish | Output to establish | Key question |
|---|---|---|---|
| Session acquisition | Stable event identifier, session/version/cache | Raw tables and provenance | What if session is missing? |
| Event audit | One driver history and timestamps | Reviewed event records | Is a pit entry a tyre change, repair or retirement? |
| Feature definition | Agreed decision boundary and historical state | Prior-known fields with units | Is it available in TUM too? |
| Quality checks | Raw records and schema | Counts, exceptions and retained cohort | Are invalid/deleted inlaps removed unfairly? |
| Race partition | Unique race identifiers | Disjoint cohort manifests | How to hold Silverstone out? |
| Model representation | Approved fields and cohort | Fixed feature order and sequence metadata | What history is actually known? |
| Simulator adapter | TUM's pre-action state and history | Comparable model representation | Are physical and effective tyre age equivalent? |

**Do not confuse an API demonstration with a validated data pipeline.** All feature transformation, labeling, splitting and integration code must be independently authored by the team. No new function implementation is claimed in this notebook.

## 7. Event worksheet: Verstappen, British GP 2019

TUM's config records inlap 13 and another inlap 20. The three-row evidence below is intentionally blank for the team to obtain from actual FastF1 data.

| Driver-lap | Lap start | Lap end | Pit entry | Pit exit | Compound | TyreLife | Stint | Position | Status |
|---|---|---|---|---|---|---|---|---|---|
| VER 12 | | | | | | | | | |
| VER 13 | | | | | | | | | |
| VER 14 | | | | | | | | | |

Write the cutoff in physical time and row notation. Identify the candidate label and evidence; note where compound mapping and tyre-age accounting remain uncertain. Compare with TUM's source entry and document any discrepancy. Check if another team member retrieves the same rows. Do not claim a successful download based on the parameter file alone.

**Review questions:** Can I state the input/output of the public API calls? When was each field known? Why is pit entry a candidate label but not a predictor? What does the new stint tell me? Which features can I also obtain at the same cut in simulation? What could make this example unrepresentative?

**Stop:** one observed event table, one proposed cutoff and one precise unresolved assumption, with source and access/reproducibility details.

## 8. What passing means

A successful live load with the expected rows verifies access to this one session. Source-schema diagnostics flag missing variables and potential data quality questions; they do not prove label correctness, temporal alignment, or generalizability across seasons. The next independently authored project task is to design and validate the agreed data contract.

## 9. Optional preprocessing from authenticated session laps

The pit target is entry during the current lap. Inputs are race progress, previous-lap position divided by the field size, and the number of earlier pit entries. The small historical preview uses the contiguous prefix through lap 14 to preserve prior state. From that preview, display laps 12–14 before constructing the full VER race table. Each example has three input features and one binary label.

In [6]:
if not RUN_LIVE:
    print("Optional session inspection skipped. The public full-race run follows.")
elif source_rows is None:
    print(f"Historical preprocessing blocked. {live_result.detail}")
else:
    preview = historical_examples(laps.loc[laps["LapNumber"] <= 14], DRIVER, 52, len(session.results))
    display(preview.loc[preview["lap"].between(12, 14)])

    examples = historical_examples(laps, DRIVER, 52, len(session.results))
    print("Silverstone feature shape:", examples[list(FEATURES)].shape)
    print("Pit-labelled laps:", examples.loc[examples["pit"] == 1, "lap"].tolist())

Optional session inspection skipped. The public full-race run follows.


## 10. Full Silverstone preprocessing with public race records

FastF1's Jolpica client provides race classification, end-of-lap positions, and pit-stop records without an F1 account. The pipeline checks each driver's lap count against the classification, joins stops by driver and lap, and reuses the shared preprocessing function. Each row uses race progress, previous position divided by field size, and prior stop count. The target is whether the driver pitted on that lap.

Lap 1 has no previous race position, so it contributes no feature row. A lap-1 stop still contributes to later prior-stop counts. The team total includes Verstappen. These counts describe the preprocessing output for one race. This feed does not supply tyre or telemetry features.

In [7]:
import pandas as pd

from f1_ai.silverstone_jolpica import preprocess_silverstone

output_dir = Path("local_results")
features, source_stops, unmatched_stops = preprocess_silverstone(
    output_dir / "fastf1_cache"
)
output_path = output_dir / "silverstone_2019_jolpica_features.csv"
features.to_csv(output_path, index=False)

ver = features.loc[features["driver"].eq("VER")]
summary = pd.DataFrame(
    {
        "Scope": ["Tony's example (VER)", "Team dataset (all drivers)"],
        "Drivers": [ver["driver"].nunique(), features["driver"].nunique()],
        "Feature rows": [len(ver), len(features)],
        "Pit-positive rows": [int(ver["pit"].sum()), int(features["pit"].sum())],
    }
).set_index("Scope")
print(f"Saved {output_path}")
print(f"Source pit-stop records: {len(source_stops)}")
print(f"Source stops on lap 1: {source_stops['lap'].eq(1).sum()}")
print(f"Stops without a matching lap record: {len(unmatched_stops)}")
display(summary)

/Users/tonys/Documents/ChatGPT/CS 682/cs682-f1-meets-ai/.venv/lib/python3.11/site-packages/fastf1/req.py:251: FutureWarning: `.enable_cache` is deprecated and will be removed in a future version. Use `.configure` instead.
  warnings.warn("`.enable_cache` is deprecated and will be removed in a "


Saved local_results/silverstone_2019_jolpica_features.csv
Source pit-stop records: 29
Source stops on lap 1: 2
Stops without a matching lap record: 0


,Drivers,Feature rows,Pit-positive rows
Scope,,,
Tony's example (VER),1,51,2
Team dataset (all drivers),20,893,27


### Tony's result · Verstappen around his two stops

Compare each pit lap with the next lap. The target changes on the pit lap, and the prior-stop count increases on the next row.

In [8]:
ver_stop_laps = ver.loc[ver["pit"].eq(1), "lap"]
print("VER pit laps:", ver_stop_laps.tolist())
display(ver.loc[ver["lap"].isin([*ver_stop_laps, *(ver_stop_laps + 1)])].round(4))

VER pit laps: [13, 20]


,driver,lap,race_progress,position_fraction,previous_pits,pit
803,VER,13,0.2308,0.20,0.0,1
804,VER,14,0.2500,0.20,1.0,0
810,VER,20,0.3654,0.25,1.0,1
811,VER,21,0.3846,0.20,2.0,0


### Team result · A preview across drivers

The table shows the first five pit-positive examples ordered by lap and driver. The CSV also includes every eligible stay-out lap.

In [9]:
display(
    features.loc[features["pit"].eq(1)]
    .sort_values(["lap", "driver"])
    .head(5)
    .round(4)
)

,driver,lap,race_progress,position_fraction,previous_pits,pit
112,GAS,12,0.2115,0.30,0.0,1
340,KVY,12,0.2115,0.75,0.0,1
599,RIC,12,0.2115,0.40,0.0,1
11,ALB,13,0.2308,0.45,0.0,1
240,HUL,13,0.2308,0.50,0.0,1
